# GAT+MLP 모델: BCE + Evidence Contrastive Loss 성능 테스트 / 하이퍼파라미터 튜닝

> **최종 업데이트**: 2026-07-14: `re_model_gat_mlp.DocREModelGATMLP` (BERT → ATLOP LCP → entity-pair rep
> → 2-layer edge-featured GAT → 2-layer MLP classifier → sigmoid, loss = BCEWithLogitsLoss +
> evi_weight × InfoNCE evidence-contrastive loss) 최초 작성. `losses.BCEEvidenceContrastiveLoss`의
> `evi_weight`/`tau`를 소규모 스윕으로 먼저 확인한 뒤, 선택한 값으로 전체 파이프라인을 학습한다.

**실행 전**: 런타임 → 런타임 유형 변경 → **GPU** 선택 (스윕은 T4로도 충분, 전체 학습은 A100 권장).

**이 노트북에서 하는 일**:
1. CPU 스모크 테스트로 배선(BERT→LCP→GAT→MLP→loss) 확인
2. `evi_weight`(0.2가 다이어그램 기본값) × `tau` 소규모 스윕 (annotated 일부 문서, 짧은 epoch)
3. threshold(시그모이드 임계값) 후보 재추론 스윕 (재학습 없이 dev로 빠르게 비교)
4. 선택된 하이퍼파라미터로 distant pretrain → annotated fine-tune 전체 학습
5. baseline(`atlop`, dev F1 61.71 / Ign F1 59.86, `results/comparison.md`)과 비교 기록

In [ ]:
# 0) GPU 확인
!nvidia-smi -L

In [ ]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별로 당김 -- LFS 대역폭 절약)
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# 아래에서 json들이 MB 단위(train_distant는 GB 근처)로 보이면 성공.
# 133바이트 같은 크기로 보이면 LFS pull 실패 -> Drive 백업본으로 우회 필요
!ls -lh docred_data/data/*.json

In [ ]:
# 2) 패키지 (Colab 기본 torch 사용, transformers 버전만 로컬 개발환경과 통일)
!pip install -q transformers==4.57.6 accelerate pandas

In [ ]:
# 3) 배선 확인: 랜덤 초기화 tiny BERT로 fwd/bwd + evidence-contrastive 항이
#    실제로 loss에 기여하는지 확인 (정확도 아님, CPU에서 수 초 내 완료)
!python -m Scripts.atlop.smoke_test_gat_mlp

## 1) `evi_weight` x `tau` 소규모 스윕

annotated 일부 문서로 짧게(3 epoch) 돌려 dev F1 경향만 비교 -- 절대 수치가 아니라 순위/방향성
확인용. `train_gat_mlp.train(args)`을 셸이 아니라 파이썬 함수로 직접 호출해서 dev F1을
리스트로 바로 모은다.

In [ ]:
import itertools
import pandas as pd

from Scripts.atlop.train_gat_mlp import train, build_gat_mlp_argparser

SWEEP_LIMIT_DOCS = 300   # annotated 3,053개 중 일부만 -- 스윕은 순위 비교가 목적
SWEEP_EPOCHS = 3

def make_args(**overrides):
    args = build_gat_mlp_argparser().parse_args([])
    args.distant_mode = "none"          # 스윕 단계는 annotated만 (distant는 4단계 전체 학습에서)
    args.limit_docs = SWEEP_LIMIT_DOCS
    args.epochs = SWEEP_EPOCHS
    args.train_batch_size = 4
    args.eval_batch_size = 8
    args.save_model = False
    for k, v in overrides.items():
        setattr(args, k, v)
    return args

sweep_rows = []
for evi_weight, tau in itertools.product([0.0, 0.2, 0.5], [0.05, 0.1, 0.2]):
    print(f"\n=== evi_weight={evi_weight}  tau={tau} ===")
    args = make_args(evi_weight=evi_weight, tau=tau,
                     run_name=f"sweep_evi{evi_weight}_tau{tau}")
    metrics = train(args)
    sweep_rows.append({"evi_weight": evi_weight, "tau": tau,
                       "dev_F1": metrics["f1"] * 100, "Ign_F1": metrics["ign_f1"] * 100,
                       "P": metrics["precision"] * 100, "R": metrics["recall"] * 100})

sweep_df = pd.DataFrame(sweep_rows).sort_values("dev_F1", ascending=False).reset_index(drop=True)
sweep_df

`evi_weight=0.0` 행은 순수 BCE(대조 항 없음) 기준선 역할 -- 다른 `(evi_weight>0, tau)` 조합이
이 행보다 낮으면 evidence-contrastive 항이 이 스윕 규모에서는 도움이 안 된다는 뜻이므로 그대로
기록해둘 것 (음의 결과도 결과).

## 2) threshold 재추론 스윕 (재학습 없음)

시그모이드 임계값은 학습된 loss에 영향을 주지 않으므로(ATLoss처럼 학습되는 TH 클래스가 아님),
위 스윕에서 고른 `(evi_weight, tau)`로 한 번만 학습한 모델을 재사용해 threshold만 바꿔가며
dev F1을 비교한다.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModel, AutoTokenizer

from data.docred_dataset import DocREDataset
from data.docred_io import build_rel2id, NUM_CLASSES
from Scripts.atlop.preprocess_full import build_features_full
from Scripts.atlop.re_model_gat_mlp import DocREModelGATMLP
from Scripts.atlop.train_full import make_collate_fn, predict
from Scripts.eval.scorer import evaluate

# 위 스윕 결과 1등 행의 evi_weight/tau로 교체하세요.
BEST_EVI_WEIGHT = float(sweep_df.iloc[0]["evi_weight"])
BEST_TAU = float(sweep_df.iloc[0]["tau"])
print(f"[threshold sweep] using evi_weight={BEST_EVI_WEIGHT} tau={BEST_TAU}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rel2id = build_rel2id(); id2rel = {v: k for k, v in rel2id.items()}
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
collate = make_collate_fn(tokenizer.pad_token_id)

train_docs = list(DocREDataset("train_annotated"))[:SWEEP_LIMIT_DOCS]
dev_docs = list(DocREDataset("dev"))[:SWEEP_LIMIT_DOCS]
dev_loader = DataLoader(build_features_full(dev_docs, tokenizer, rel2id),
                        batch_size=8, shuffle=False, collate_fn=collate)

config = AutoConfig.from_pretrained("bert-base-cased", num_labels=NUM_CLASSES)
encoder = AutoModel.from_pretrained("bert-base-cased", config=config, attn_implementation="eager")
config.cls_token_id, config.sep_token_id = tokenizer.cls_token_id, tokenizer.sep_token_id
model = DocREModelGATMLP(config, encoder, num_labels=NUM_CLASSES,
                        evi_weight=BEST_EVI_WEIGHT, tau=BEST_TAU).to(device)

threshold_rows = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    model.threshold = threshold
    preds = predict(model, dev_loader, id2rel, device)
    metrics = evaluate(preds, dev_docs, train_docs)
    threshold_rows.append({"threshold": threshold, "dev_F1": metrics["f1"] * 100,
                           "Ign_F1": metrics["ign_f1"] * 100,
                           "P": metrics["precision"] * 100, "R": metrics["recall"] * 100})

threshold_df = pd.DataFrame(threshold_rows).sort_values("dev_F1", ascending=False).reset_index(drop=True)
threshold_df

**참고**: 이 threshold 셀은 위에서 방금 만든 랜덤 초기화 모델로 도는 것이라 수치 자체는 의미
없음(배선 확인용 패턴) -- 실제로 threshold를 고르려면 3단계 학습이 끝난 뒤 같은 셀을
체크포인트를 로드해서(`model.load_state_dict(torch.load(...))`) 다시 돌릴 것.

## 3) 전체 학습 (distant pretrain -> annotated fine-tune)

위에서 고른 `evi_weight`/`tau`/`threshold`로 교체 후 실행 (A100 기준 약 1~2시간,
`--distant_limit`으로 조절).

In [ ]:
# 아래 세 값을 위 스윕에서 고른 값으로 바꾸세요.
EVI_WEIGHT = 0.2
TAU = 0.1
THRESHOLD = 0.5

!python -m Scripts.atlop.train_gat_mlp --epochs 15 --distant_limit 20000 --distant_epochs 1 \
  --eval_batch_size 32 --evi_weight {EVI_WEIGHT} --tau {TAU} --threshold {THRESHOLD} \
  --run_name atlop_gat_mlp --save_model --seed 66

In [ ]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/atlop_gat_mlp.pt results/atlop_gat_mlp_stage1.pt \
   results/atlop_gat_mlp_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"

## 결과 기록

학습이 끝나면 마지막 epoch 로그의 수치를 아래 표에 채워서 `results/comparison.md`에 반영:

| | dev F1 | Ign F1 | P | R |
|---|---|---|---|---|
| baseline (ATLoss + bilinear, 팀원 `atlop`) | 61.71 | 59.86 | 66.08 | 57.89 |
| **GAT+MLP+Sigmoid (BCE + evi_weight x contrastive, 이 노트북)** | | | | |

- 위 하이퍼파라미터 스윕 표(`sweep_df`, `threshold_df`)도 함께 캡처해두면 "왜 이 값을
  골랐는지" 재현 가능.
- seed 66 단일 시드 결과이므로, baseline과 차이가 ±1점 이내면 PRD §5의 시드 노이즈 주의사항 적용.